In [0]:
"""
05_operator_dimension.py

Operator Dimension (SCD Type 1)

Source:
    operation_events

Target:
    operator_dimension

Author:
Sumanth Vempalle

Version:
2.2.0
"""

import dlt

from pyspark.sql.functions import (
    col,
)

# ============================================================
# Operator Source View
# ============================================================

@dlt.view(
    name="operator_dimension_source",
    comment="Source view for Operator Dimension."
)
def operator_dimension_source():

    return (

        spark.readStream.table(
            "operation_events"
        )

        .select(

            col("operator_id"),

            col("operator_name"),

            col("skill_level"),

            col("department"),

            col("event_timestamp").alias(
                "last_updated"
            ),

        )

        .filter(
            col("operator_id").isNotNull()
        )

        .dropDuplicates(
            [
                "operator_id"
            ]
        )

    )


# ============================================================
# Target Streaming Table
# ============================================================

dlt.create_streaming_table(

    name="operator_dimension",

    comment="Operator Dimension (SCD Type 1)."

)


# ============================================================
# AUTO CDC FLOW
# ============================================================

dlt.create_auto_cdc_flow(

    target="operator_dimension",

    source="operator_dimension_source",

    keys=[
        "operator_id",
    ],

    sequence_by="last_updated",

    stored_as_scd_type=1,

)